# 01 — Data Cleaning (VaxFlow)

ทำความสะอาดข้อมูลโดเมนวัคซีน + **สังเคราะห์ชุดข้อมูลความต้องการเชิงสุ่ม** 
(Stochastic Demand Pattern) ตาม Proposal §4.1 เพื่อใช้เป็นฐานพยากรณ์ดีมานด์

อินพุต: `data/vaccine/*.csv`, `data/hospitals/hospital_master.csv`  
เอาต์พุต: `data/vaccine/clean/{demand_daily,vaccine_vial_clean,vaccine_product_clean}.csv`

In [11]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
VAX = ROOT / 'data' / 'vaccine'
CLEAN = VAX / 'clean'
CLEAN.mkdir(parents=True, exist_ok=True)
print('data dir:', VAX.resolve())

data dir: C:\Work\LogisticsInnovationHackathon2026\VaxFlow\data\vaccine


## 1) โหลดข้อมูลดิบ + ตรวจความถูกต้อง

In [12]:
products = pd.read_csv(VAX / 'vaccine_product.csv')
vials = pd.read_csv(VAX / 'vaccine_vial.csv', parse_dates=['state_since', 'effective_expiry'])
queue = pd.read_csv(VAX / 'appointment_queue.csv', parse_dates=['queue_date'])
branches = pd.read_csv(VAX / 'vaccine_branches.csv')
master = pd.read_csv(ROOT / 'data' / 'hospitals' / 'hospital_master.csv')
for name, df in [('product', products), ('vial', vials), ('queue', queue), ('branches', branches)]:
    print(f'{name:9s}: {df.shape[0]:>4} rows x {df.shape[1]} cols')

product  :    2 rows x 7 cols
vial     :  113 rows x 9 cols
queue    :  140 rows x 4 cols
branches :    5 rows x 2 cols


In [13]:
# ตรวจ integrity: state ที่ถูกต้อง, โดสไม่ติดลบ, FK product/hospital ครบ
VALID_STATES = {'DEEP_FROZEN', 'THAWED', 'OPENED'}
assert set(vials['state']).issubset(VALID_STATES), 'พบ state ไม่ถูกต้อง'
assert (vials['doses_remaining'] >= 0).all(), 'พบ doses_remaining ติดลบ'
assert vials['product_id'].isin(products['product_id']).all(), 'พบ product_id ที่ไม่มีใน master'
vials = vials.drop_duplicates('vial_id').reset_index(drop=True)
print('vial integrity OK ·', vials['state'].value_counts().to_dict())

vial integrity OK · {'DEEP_FROZEN': 72, 'THAWED': 32, 'OPENED': 9}


## 2) สังเคราะห์ดีมานด์รายวัน (Stochastic Demand Pattern)

180 วันย้อนหลัง × 5 สาขา × 2 ผลิตภัณฑ์ — มี **แนวโน้ม + ฤดูกาลรายสัปดาห์ + noise** 
และ **ช่วงวิกฤต (demand spike)** เพื่อใช้ทดสอบทั้งสภาวะปกติและวิกฤต (Proposal §4.3)

In [14]:
BRANCHES = sorted(branches['hospital_id'].unique())
PRODUCTS = list(products['product_id'])
N_DAYS = 180
END = pd.Timestamp('2026-06-25')
dates = pd.date_range(END - pd.Timedelta(days=N_DAYS - 1), END, freq='D')
rng = np.random.default_rng(2026)

rows = []
for hi, hid in enumerate(BRANCHES):
    branch_scale = rng.uniform(0.7, 1.6)            # ขนาดสาขาต่างกัน
    for pid in PRODUCTS:
        base = branch_scale * rng.uniform(8, 20)
        trend = np.linspace(0, rng.uniform(-3, 5), N_DAYS)    # แนวโน้มเพิ่ม/ลด
        weekday = np.array([1.25 if d.weekday() < 5 else 0.6 for d in dates])  # วันทำการสูงกว่า
        noise = rng.normal(1.0, 0.18, N_DAYS)
        demand = (base + trend) * weekday * noise
        # ช่วงวิกฤต: สุ่ม 1-2 ช่วง spike (เช่น การระบาด/แคมเปญ)
        for _ in range(rng.integers(1, 3)):
            s = rng.integers(0, N_DAYS - 14); demand[s:s + 10] *= rng.uniform(1.8, 2.8)
        demand = np.clip(np.round(demand), 0, None).astype(int)
        rows.append(pd.DataFrame({'date': dates, 'hospital_id': hid,
                                  'product_id': pid, 'demand': demand}))
demand_daily = pd.concat(rows, ignore_index=True)
print('demand_daily:', demand_daily.shape, '· total doses demanded:', int(demand_daily['demand'].sum()))
demand_daily.head()

demand_daily: (1800, 4) · total doses demanded: 36864


,date,hospital_id,product_id,demand
0,2025-12-28,HOSP_001,VAX_MRNA_01,10
1,2025-12-29,HOSP_001,VAX_MRNA_01,45
2,2025-12-30,HOSP_001,VAX_MRNA_01,38
3,2025-12-31,HOSP_001,VAX_MRNA_01,38
4,2026-01-01,HOSP_001,VAX_MRNA_01,42


## 3) บันทึกข้อมูลที่สะอาดแล้ว

In [15]:
demand_daily.to_csv(CLEAN / 'demand_daily.csv', index=False, encoding='utf-8-sig')
vials.to_csv(CLEAN / 'vaccine_vial_clean.csv', index=False, encoding='utf-8-sig')
products.to_csv(CLEAN / 'vaccine_product_clean.csv', index=False, encoding='utf-8-sig')
print('[saved] ->', CLEAN.resolve())
for f in sorted(CLEAN.glob('*.csv')):
    print(' ', f.name)

[saved] -> C:\Work\LogisticsInnovationHackathon2026\VaxFlow\data\vaccine\clean
  demand_daily.csv
  vaccine_product_clean.csv
  vaccine_vial_clean.csv
